# Data/MC agreement summary over all BDT input variables

Reads `plots/all_p_value_info.pkl` written by `python src/plot_many_variables.py` (WC generic preselection, open-data prediction vs data).

- **No DetVar** / **With DetVar**: covariance without / with detector-variation systematics
- **Full χ²**: χ² over all bins, ndf = number of bins
- **Global decomp**: largest |ε_i| from the χ² eigen-decomposition, trials-corrected for the number of components, p_global = 1 − (1 − p_local)^N

The variables are strongly correlated, so their p-values are not independent and the uniform expectation is only a rough guide.

Plots are saved as PDF and PNG in `plots/bdt_var_p_values/`.

In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2, norm

plt.rcParams.update({"font.size": 12})

out_dir = Path("../plots/bdt_var_p_values")
out_dir.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    fig.savefig(out_dir / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(out_dir / f"{name}.png", bbox_inches="tight", dpi=200)

## Load, and recompute p-values and σ with full precision

The stored values use `1 - chi2.cdf(...)`, which underflows to p = 0 (σ = ∞) for large χ². They are recomputed from the stored χ²/ndf and max local σ using log survival functions. The convention is the same two-sided one, σ = Φ⁻¹(1 − p/2).

In [ ]:
with open("../plots/all_p_value_info.pkl", "rb") as f:
    df = pd.DataFrame(pickle.load(f))
print(f"{len(df)} variables")

def logp_to_sigma(logp):
    return norm.isf(np.exp(logp - np.log(2)))

for cov in ["nodetvar", "tot"]:
    logp = chi2.logsf(df[f"{cov}_chi2"], df["ndf"])
    df[f"{cov}_p"] = np.exp(logp)
    df[f"{cov}_sig"] = logp_to_sigma(logp)

    # global: max local |eps| (1 dof), trials-corrected over ndf components; log(1 - (1 - p)^N) computed stably
    local_logp = chi2.logsf(df[f"{cov}_max_local_sigma"] ** 2, 1)
    global_p = -np.expm1(df["ndf"] * np.log1p(-np.exp(local_logp)))
    global_logp = np.where(global_p < 1e-12, np.log(df["ndf"]) + local_logp, np.log(np.clip(global_p, 1e-300, None)))
    df[f"{cov}_global_p"] = np.exp(global_logp)
    df[f"{cov}_global_sig"] = logp_to_sigma(global_logp)

    # cross-check against the stored values where the old 1 - cdf is still precise
    precise = df[f"{cov}_p"] > 1e-10
    assert np.allclose(df.loc[precise, f"{cov}_sig"], df.loc[precise, f"{cov}_sigma"], rtol=1e-6, atol=1e-6)

tests = {
    "nodetvar": ("No DetVar, full $\\chi^2$", "tab:blue", "-"),
    "nodetvar_global": ("No DetVar, global decomp", "tab:blue", "--"),
    "tot": ("With DetVar, full $\\chi^2$", "tab:orange", "-"),
    "tot_global": ("With DetVar, global decomp", "tab:orange", "--"),
}

## p-value and σ distributions

Each plot is made twice: with all four tests, and with only the With DetVar curves (files ending in `_with_detvar_only`).

In [ ]:
with_detvar_tests = {k: v for k, v in tests.items() if k.startswith("tot")}

def plot_p_values(tests_to_plot, name):
    bins = np.linspace(0, 1, 26)
    fig, ax = plt.subplots(figsize=(10, 6))
    for key, (lab, color, ls) in tests_to_plot.items():
        ax.hist(df[f"{key}_p"], bins=bins, histtype="step", label=lab, color=color, linestyle=ls, lw=1.5)
    ax.axhline(len(df) / (len(bins) - 1), color="k", linestyle=":", label="Uniform expectation")
    ax.set_xlabel("p-value")
    ax.set_ylabel("Variables")
    ax.set_yscale("log")
    ax.set_title(f"All {len(df)} BDT variables, WC generic preselection")
    ax.legend(fontsize=10)
    save(fig, name)
    plt.show()

def plot_sigmas(tests_to_plot, name):
    max_sig = max(df[f"{key}_sig"].max() for key in tests_to_plot)
    bins = np.arange(0, np.ceil(max_sig) + 0.25, 0.25)
    fig, ax = plt.subplots(figsize=(10, 6))
    max_count = 0
    for key, (lab, color, ls) in tests_to_plot.items():
        counts, _, _ = ax.hist(df[f"{key}_sig"], bins=bins, histtype="step", label=lab, color=color, linestyle=ls, lw=1.5)
        max_count = max(max_count, counts.max())
    # expected for uniform p with the two-sided convention: sigma ~ |N(0,1)|
    expected_counts = len(df) * 2 * (norm.sf(bins[:-1]) - norm.sf(bins[1:]))
    ax.step(bins[:-1], expected_counts, where="post", color="k", linestyle=":", label="Expected for uniform p")
    ax.set_xlabel(r"$\sigma$")
    ax.set_ylabel("Variables")
    ax.set_yscale("log")
    ax.set_ylim(0.5, max_count * 3)
    ax.set_title(f"All {len(df)} BDT variables, WC generic preselection")
    ax.legend(fontsize=10)
    save(fig, name)
    plt.show()

In [ ]:
plot_p_values(tests, "p_value_distribution")
plot_p_values(with_detvar_tests, "p_value_distribution_with_detvar_only")

In [ ]:
plot_sigmas(tests, "sigma_distribution")
plot_sigmas(with_detvar_tests, "sigma_distribution_with_detvar_only")

## Most discrepant variables (With DetVar, full χ²)

In [ ]:
sigma_threshold = 2
discrepant = df[df["tot_sig"] > sigma_threshold].sort_values("tot_sig", ascending=False).reset_index(drop=True)
show_cols = ["var", "ndf", "tot_chi2", "tot_sig", "tot_global_sig", "nodetvar_chi2", "nodetvar_sig", "nodetvar_global_sig"]
print(f"{len(discrepant)} variables above {sigma_threshold} sigma (expected for uniform, independent p: {len(df) * 2 * norm.sf(sigma_threshold):.1f})")
pd.set_option("display.max_colwidth", 60)
discrepant[show_cols].round(2)

## Combined PDF of the discrepant variables, sorted by descending σ

Re-runs `src/plot_many_variables.py` on just the discrepant variables, in descending-σ order, with two extra columns on each page: the systematic and DetVar breakdowns, and each DetVar variation's spectrum with its (Var − CV)/CV fractional difference. The script runs as a separate process so that its large data loads stay out of this kernel. It writes:
- the individual pages to `plots/discrepant_bdt_vars/`
- the combined PDF to `plots/discrepant_bdt_vars.pdf`
- the p-value info to `plots/discrepant_bdt_vars_p_value_info.pkl`

It does not touch `all_p_value_info.pkl`. The systematic covariances come from the cache filled by the full run, so this is much faster than the full run.

Set `run_plots = False` to skip re-plotting and only check an existing output.

In [ ]:
import subprocess
import sys

run_plots = True
output_name = "discrepant_bdt_vars"

if run_plots:
    cmd = [sys.executable, "src/plot_many_variables.py",
           "--vars", ",".join(discrepant["var"]),
           "--output-name", output_name,
           "--sys-breakdown-panels",
           "--detvar-variation-panels",
           "--clear-prev"]
    proc = subprocess.Popen(cmd, cwd="..", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    log_lines = []
    for line in proc.stdout:
        log_lines.append(line)
        # only echo progress lines, the full log is kept in log_lines
        if line.startswith(("plotting", "skipping", "Final PDF", "WARNING", "ERROR", "Traceback")):
            print(line, end="")
    if proc.wait() != 0:
        print("".join(log_lines[-50:]))
        raise RuntimeError(f"plot_many_variables.py failed with exit code {proc.returncode}")

In [ ]:
# the new run's chi2 values should reproduce the ones from the full run
with open(f"../plots/{output_name}_p_value_info.pkl", "rb") as f:
    rerun_df = pd.DataFrame(pickle.load(f))
assert rerun_df["var"].tolist() == discrepant["var"].tolist(), "page order differs from the sigma ordering"
check = discrepant[["var", "tot_chi2", "nodetvar_chi2"]].merge(rerun_df[["var", "tot_chi2", "nodetvar_chi2"]], on="var", suffixes=("", "_rerun"))
for cov in ["tot", "nodetvar"]:
    max_rel_diff = (np.abs(check[f"{cov}_chi2_rerun"] - check[f"{cov}_chi2"]) / check[f"{cov}_chi2"]).max()
    print(f"{cov} chi2: max relative difference vs full run = {max_rel_diff:.2e}")
print(f"{len(rerun_df)} pages in {Path(f'../plots/{output_name}.pdf').resolve()}")